In [ ]:
# pip install -U langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. STATE: the shared memory every node reads from and writes to
class State(TypedDict):
    topic: str
    draft: str
    attempts: int
    approved: bool


# 2. NODES: plain functions. Take state in, return ONLY the fields you changed
def write(state: State) -> dict:
    attempts = state["attempts"] + 1
    draft = f"Draft v{attempts} about {state['topic']}" + " (more detail)" * attempts
    print(f"write  -> attempt {attempts}, {len(draft)} chars")
    return {"draft": draft, "attempts": attempts}


def review(state: State) -> dict:
    approved = len(state["draft"]) > 50  # fake "quality check"
    print(f"review -> approved={approved}")
    return {"approved": approved}


# 3. ROUTER: decides which edge to follow next (this is the "conditional edge")
def route(state: State) -> str:
    if state["approved"] or state["attempts"] >= 5:  # hard stop prevents infinite loops
        return "done"
    return "retry"


# 4. BUILD THE GRAPH
builder = StateGraph(State)
builder.add_node("write", write)
builder.add_node("review", review)

builder.add_edge(START, "write")          # fixed edge
builder.add_edge("write", "review")       # fixed edge
builder.add_conditional_edges(            # decision point
    "review", route, {"retry": "write", "done": END}
)

graph = builder.compile()                 # validates + turns it into a runnable

# 5. RUN
result = graph.invoke(
    {"topic": "LangGraph", "draft": "", "attempts": 0, "approved": False}
)
print(result)

write  -> attempt 1, 38 chars
review -> approved=False
write  -> attempt 2, 52 chars
review -> approved=True
{'topic': 'LangGraph', 'draft': 'Draft v2 about LangGraph (more detail) (more detail)', 'attempts': 2, 'approved': True}
